# TCN data preparation — Las Vegas and Phoenix electricity demand

This notebook prepares hourly weather, calendar, and historical-load inputs for a **one-hour-ahead** Temporal Convolutional Network (TCN). It does **not** train a model.

The supplied files were inspected before this notebook was written: each energy file has `timestamp` and `hourly_load_mw`; each weather file has `timestamp` plus weather variables. The weather files have 48 fewer hourly rows than the energy files, so the pipeline restores a complete hourly index and treats absent weather observations carefully.

## Design choices

- **Forecast horizon:** inputs through hour *t* predict load at *t + 1*.
- **Lookback:** 168 hours (one week) by default, adjustable below.
- **Chronology:** 70% train / 15% validation / 15% test, with no shuffling.
- **Leakage prevention:** rolling load features use `shift(1)`; scalers fit only on training rows; a sequence is assigned according to its target timestamp.
- **Weather gaps:** invalid readings become missing, then use forward-fill only (past information). Rows still incomplete are excluded from sequences rather than filled using future information.

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Windows-friendly paths. Change only DATA_DIR if your folder moves.
DATA_DIR = (Path.cwd().parent / "data")
OUTPUT_DIR = DATA_DIR / "tcn_prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 168          # one week of hourly history
HORIZON = 1             # predict load one hour ahead
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
RANDOM_SEED = 42
assert 0 < TRAIN_FRACTION < 1 and 0 < VALIDATION_FRACTION < 1
assert TRAIN_FRACTION + VALIDATION_FRACTION < 1
assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print(f"Inputs:  {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

Inputs:  D:\Downloads\BTP\model_training\lstm_training_3\data
Outputs: D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_prepared


In [2]:
# Discover and match files instead of relying only on hard-coded names.
csv_files = sorted(DATA_DIR.glob("*.csv"))
assert csv_files, f"No CSV files found in {DATA_DIR}"

def kind_from_name(path):
    name = path.stem.lower()
    if "weather" in name:
        return "weather"
    if "energy" in name or "load" in name:
        return "energy"
    return None

file_map = {}
for path in csv_files:
    file_kind = kind_from_name(path)
    if file_kind is None:
        continue
    city = path.stem.lower().split("_hourly_")[0]
    file_map.setdefault(city, {})[file_kind] = path

assert file_map, "Could not identify weather/energy CSV files from their names."
for city, files in file_map.items():
    assert {"energy", "weather"} <= files.keys(), f"{city} is missing a weather or energy file: {files}"
    print(f"{city}:\n  energy  = {files['energy'].name}\n  weather = {files['weather'].name}")

# This should list Las Vegas and Phoenix in the supplied dataset.
assert {"las_vegas", "phoenix"} <= file_map.keys(), "Expected Las Vegas and Phoenix files were not found."

las_vegas:
  energy  = las_vegas_hourly_energy.csv
  weather = las_vegas_hourly_weather_2016_2023.csv
phoenix:
  energy  = phoenix_hourly_energy.csv
  weather = phoenix_hourly_weather_2016_2023.csv


In [3]:
# Inspect the actual schemas before processing.
for city, files in file_map.items():
    print(f"\n{city.upper()}")
    for file_kind, path in files.items():
        preview = pd.read_csv(path, nrows=3)
        print(f"{file_kind}: {len(preview.columns)} columns -> {preview.columns.tolist()}")
        display(preview.head(2))


LAS_VEGAS
energy: 2 columns -> ['timestamp', 'hourly_load_mw']


,timestamp,hourly_load_mw
0,2016-01-01 00:00:00,2792.310974
1,2016-01-01 01:00:00,2975.455849


weather: 25 columns -> ['date', 'hour', 'extraterrestrial_horizontal_radiation_wh_m2', 'extraterrestrial_direct_normal_radiation_wh_m2', 'global_horizontal_radiation_wh_m2', 'direct_normal_radiation_wh_m2', 'diffuse_horizontal_radiation_wh_m2', 'total_sky_cover_tenths', 'opaque_sky_cover_tenths', 'temperature_c', 'dew_point_c', 'relative_humidity_percent', 'station_pressure_kpa', 'wind_direction_deg', 'wind_speed_ms', 'horizontal_visibility_km', 'ceiling_height_m', 'observation_indicator', 'present_weather', 'precipitable_water_mm', 'broadband_aerosol_optical_depth', 'snow_depth_cm', 'days_since_last_snowfall', 'hourly_precipitation_cm', 'timestamp']


,date,hour,extraterrestrial_horizontal_radiation_wh_m2,extraterrestrial_direct_normal_radiation_wh_m2,global_horizontal_radiation_wh_m2,direct_normal_radiation_wh_m2,diffuse_horizontal_radiation_wh_m2,total_sky_cover_tenths,opaque_sky_cover_tenths,temperature_c,...,horizontal_visibility_km,ceiling_height_m,observation_indicator,present_weather,precipitable_water_mm,broadband_aerosol_optical_depth,snow_depth_cm,days_since_last_snowfall,hourly_precipitation_cm,timestamp
0,2016-01-01,1,0.0,0.0,NaN,NaN,NaN,0.0,0.0,-1.1,...,24.1,10395.0,0,999999999,5.0,0.107,0.0,88.0,0.0,2016-01-01 00:00:00
1,2016-01-01,2,0.0,0.0,NaN,NaN,NaN,0.0,0.0,-0.6,...,24.1,10395.0,0,999999999,5.0,0.107,0.0,88.0,0.0,2016-01-01 01:00:00



PHOENIX
energy: 2 columns -> ['timestamp', 'hourly_load_mw']


,timestamp,hourly_load_mw
0,2016-01-01 00:00:00,4501.272684
1,2016-01-01 01:00:00,4981.164660


weather: 25 columns -> ['date', 'hour', 'extraterrestrial_horizontal_radiation_wh_m2', 'extraterrestrial_direct_normal_radiation_wh_m2', 'global_horizontal_radiation_wh_m2', 'direct_normal_radiation_wh_m2', 'diffuse_horizontal_radiation_wh_m2', 'total_sky_cover_tenths', 'opaque_sky_cover_tenths', 'temperature_c', 'dew_point_c', 'relative_humidity_percent', 'station_pressure_kpa', 'wind_direction_deg', 'wind_speed_ms', 'horizontal_visibility_km', 'ceiling_height_m', 'observation_indicator', 'present_weather', 'precipitable_water_mm', 'broadband_aerosol_optical_depth', 'snow_depth_cm', 'days_since_last_snowfall', 'hourly_precipitation_cm', 'timestamp']


,date,hour,extraterrestrial_horizontal_radiation_wh_m2,extraterrestrial_direct_normal_radiation_wh_m2,global_horizontal_radiation_wh_m2,direct_normal_radiation_wh_m2,diffuse_horizontal_radiation_wh_m2,total_sky_cover_tenths,opaque_sky_cover_tenths,temperature_c,...,horizontal_visibility_km,ceiling_height_m,observation_indicator,present_weather,precipitable_water_mm,broadband_aerosol_optical_depth,snow_depth_cm,days_since_last_snowfall,hourly_precipitation_cm,timestamp
0,2016-01-01,1,0.0,0.0,NaN,NaN,NaN,0.0,0.0,7.2,...,56.3,16764.0,0,999999999,8.0,0.178,0.0,88.0,0.0,2016-01-01 00:00:00
1,2016-01-01,2,0.0,0.0,NaN,NaN,NaN,0.0,0.0,6.1,...,56.3,16764.0,0,999999999,8.0,0.178,0.0,88.0,0.0,2016-01-01 01:00:00


In [4]:
TARGET = "hourly_load_mw"

# Plausible physical ranges. Values outside these ranges are treated as invalid.
# Coded fields such as present_weather and observation_indicator are intentionally excluded.
VALID_RANGES = {
    "temperature_c": (-60, 60), "dew_point_c": (-80, 50),
    "relative_humidity_percent": (0, 100), "station_pressure_kpa": (80, 110),
    "wind_direction_deg": (0, 360), "wind_speed_ms": (0, 75),
    "horizontal_visibility_km": (0, 100), "ceiling_height_m": (0, 30000),
    "precipitable_water_mm": (0, 100), "hourly_precipitation_cm": (0, 100),
    "total_sky_cover_tenths": (0, 10), "opaque_sky_cover_tenths": (0, 10),
    "extraterrestrial_horizontal_radiation_wh_m2": (0, np.inf),
    "extraterrestrial_direct_normal_radiation_wh_m2": (0, np.inf),
}

WEATHER_BASE = list(VALID_RANGES)

def read_hourly_csv(path, required_columns):
    df = pd.read_csv(path)
    assert "timestamp" in df.columns, f"timestamp missing from {path.name}"
    missing = set(required_columns) - set(df.columns)
    assert not missing, f"{path.name} is missing required columns: {missing}"
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    assert df["timestamp"].notna().all(), f"Unparseable timestamps in {path.name}"
    df = df.sort_values("timestamp").drop_duplicates("timestamp", keep="last")
    return df.reset_index(drop=True)

def clean_weather(weather):
    available = [c for c in WEATHER_BASE if c in weather.columns]
    weather = weather[["timestamp"] + available].copy()
    for col in available:
        weather[col] = pd.to_numeric(weather[col], errors="coerce")
        low, high = VALID_RANGES[col]
        weather.loc[~weather[col].between(low, high), col] = np.nan
    # Full hourly grid exposes absent weather timestamps. ffill never uses future data.
    weather = weather.set_index("timestamp").asfreq("h")
    weather[available] = weather[available].ffill()
    return weather.reset_index(), available

def add_features(df, weather_columns):
    out = df.copy().sort_values("timestamp").reset_index(drop=True)
    ts = out["timestamp"]
    # Cyclic calendar variables avoid treating Sunday and Monday / 23:00 and 00:00 as far apart.
    out["hour_sin"] = np.sin(2 * np.pi * ts.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * ts.dt.hour / 24)
    out["dow_sin"] = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    out["month_sin"] = np.sin(2 * np.pi * (ts.dt.month - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (ts.dt.month - 1) / 12)
    out["is_weekend"] = (ts.dt.dayofweek >= 5).astype("int8")
    out["is_holiday"] = 0  # Replace with a verified US holiday calendar if available.

    if "wind_direction_deg" in out:
        radians = np.deg2rad(out["wind_direction_deg"])
        out["wind_dir_sin"] = np.sin(radians)
        out["wind_dir_cos"] = np.cos(radians)
        weather_columns = [c for c in weather_columns if c != "wind_direction_deg"]
        weather_columns += ["wind_dir_sin", "wind_dir_cos"]
    if "temperature_c" in out:
        out["cooling_degree_c"] = np.clip(out["temperature_c"] - 18.0, 0, None)
        out["heating_degree_c"] = np.clip(18.0 - out["temperature_c"], 0, None)
        weather_columns += ["cooling_degree_c", "heating_degree_c"]
    if "relative_humidity_percent" in out and "temperature_c" in out:
        out["temp_x_humidity"] = out["temperature_c"] * out["relative_humidity_percent"]
        weather_columns += ["temp_x_humidity"]

    # Every load-derived feature is shifted first: at t it contains values strictly before t.
    out["load_lag_0h"] = out[TARGET]  # load observed at the input hour; never the future target
    out["load_lag_1h"] = out[TARGET].shift(1)
    out["load_lag_24h"] = out[TARGET].shift(24)
    out["load_lag_168h"] = out[TARGET].shift(168)
    out["load_roll_mean_24h"] = out[TARGET].shift(1).rolling(24, min_periods=24).mean()
    out["load_roll_std_24h"] = out[TARGET].shift(1).rolling(24, min_periods=24).std()
    out["load_roll_mean_168h"] = out[TARGET].shift(1).rolling(168, min_periods=168).mean()
    # The sequence ending at timestamp t predicts this target at t + HORIZON.
    # It is kept at its own (target) timestamp to make split assignment unambiguous.
    out["target_t_plus_1"] = out[TARGET]

    calendar = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos", "is_weekend", "is_holiday"]
    load_history = ["load_lag_0h", "load_lag_1h", "load_lag_24h", "load_lag_168h", "load_roll_mean_24h", "load_roll_std_24h", "load_roll_mean_168h"]
    feature_columns = list(dict.fromkeys(weather_columns + calendar + load_history))
    return out, feature_columns


In [5]:
def make_sequences(X, y, timestamps, target_positions, lookback):
    """Build (samples, lookback, features) windows ending one hour before each target."""
    sequences, targets, target_times = [], [], []
    valid_rows = np.isfinite(X).all(axis=1) & np.isfinite(y)
    for target_pos in target_positions:
        input_end = target_pos - HORIZON
        input_start = input_end - lookback + 1
        if input_start < 0:
            continue
        if valid_rows[input_start:target_pos + 1].all():
            sequences.append(X[input_start:input_end + 1])
            targets.append(y[target_pos])
            target_times.append(timestamps[target_pos])
    return np.asarray(sequences, dtype=np.float32), np.asarray(targets, dtype=np.float32), pd.DatetimeIndex(target_times)

def prepare_city(city, files):
    energy = read_hourly_csv(files["energy"], [TARGET])[["timestamp", TARGET]]
    energy[TARGET] = pd.to_numeric(energy[TARGET], errors="coerce")
    energy.loc[energy[TARGET] <= 0, TARGET] = np.nan
    # Energy provides the complete required timeline. Never invent an absent target.
    energy = energy.set_index("timestamp").asfreq("h").reset_index()
    weather = read_hourly_csv(files["weather"], [])
    weather, weather_columns = clean_weather(weather)
    merged = energy.merge(weather, on="timestamp", how="left", validate="one_to_one")
    featured, feature_columns = add_features(merged, weather_columns)

    n = len(featured)
    train_end = int(n * TRAIN_FRACTION)
    valid_end = int(n * (TRAIN_FRACTION + VALIDATION_FRACTION))
    split_bounds = {"train": (0, train_end), "validation": (train_end, valid_end), "test": (valid_end, n)}

    # Fit both scalers only on train rows with complete feature/target observations.
    train_frame = featured.iloc[:train_end]
    scaler_fit_mask = train_frame[feature_columns + ["target_t_plus_1"]].notna().all(axis=1)
    assert scaler_fit_mask.any(), f"No complete training rows for {city}"
    feature_scaler = StandardScaler().fit(train_frame.loc[scaler_fit_mask, feature_columns])
    target_scaler = StandardScaler().fit(train_frame.loc[scaler_fit_mask, ["target_t_plus_1"]])

    # StandardScaler preserves NaNs; unavailable observations invalidate the affected sequence.
    X = feature_scaler.transform(featured[feature_columns])
    y_raw = featured["target_t_plus_1"].to_numpy(dtype=float)
    y = np.full(n, np.nan, dtype=float)
    target_ok = np.isfinite(y_raw)
    y[target_ok] = target_scaler.transform(y_raw[target_ok].reshape(-1, 1)).ravel()

    arrays, split_summary = {}, {}
    for split_name, (start, end) in split_bounds.items():
        positions = range(start, end)
        X_seq, y_seq, target_times = make_sequences(X, y, featured["timestamp"].to_numpy(), positions, LOOKBACK)
        assert X_seq.ndim == 3 or len(X_seq) == 0
        assert len(X_seq) == len(y_seq) == len(target_times)
        arrays[split_name] = (X_seq, y_seq, target_times)
        split_summary[split_name] = {
            "target_start": str(featured["timestamp"].iloc[start]),
            "target_end": str(featured["timestamp"].iloc[end - 1]),
            "samples": int(len(y_seq)),
        }

    city_dir = OUTPUT_DIR / city
    city_dir.mkdir(exist_ok=True)
    for split_name, (X_seq, y_seq, target_times) in arrays.items():
        np.save(city_dir / f"X_{split_name}.npy", X_seq)
        np.save(city_dir / f"y_{split_name}.npy", y_seq)
        pd.DataFrame({"target_timestamp": target_times}).to_csv(city_dir / f"{split_name}_target_timestamps.csv", index=False)
    joblib.dump(feature_scaler, city_dir / "feature_scaler.joblib")
    joblib.dump(target_scaler, city_dir / "target_scaler.joblib")
    featured.to_csv(city_dir / "featured_hourly_data.csv", index=False)

    metadata = {
        "city": city, "target": TARGET, "forecast_horizon_hours": HORIZON, "lookback_hours": LOOKBACK,
        "feature_columns": feature_columns, "raw_rows": int(n), "splits": split_summary,
        "feature_scaler_fit_rows": int(scaler_fit_mask.sum()),
        "source_files": {k: str(v) for k, v in files.items()},
    }
    with open(city_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return featured, feature_columns, arrays, metadata


In [6]:
# Run preparation for each discovered city.
results = {}
for city, files in file_map.items():
    print(f"Preparing {city}...")
    results[city] = prepare_city(city, files)
print("Preparation complete.")

Preparing las_vegas...


d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Preparing phoenix...


d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Preparation complete.


In [7]:
# Validation checks: shapes, chronology, missing values, target alignment, and split/scaler leakage.
for city, (featured, feature_columns, arrays, metadata) in results.items():
    print(f"\n{city.upper()}")
    print(f"Hourly rows: {len(featured):,}; features: {len(feature_columns)}")
    for split_name, (X_seq, y_seq, target_times) in arrays.items():
        assert X_seq.ndim == 3, f"{city}/{split_name}: X must be rank 3"
        assert X_seq.shape[1:] == (LOOKBACK, len(feature_columns))
        assert len(X_seq) == len(y_seq) == len(target_times)
        assert np.isfinite(X_seq).all(), f"{city}/{split_name}: NaN/inf in X"
        assert np.isfinite(y_seq).all(), f"{city}/{split_name}: NaN/inf in y"
        assert target_times.is_monotonic_increasing
        print(f"  {split_name:10s} X={X_seq.shape}, y={y_seq.shape}, targets={target_times.min()} to {target_times.max()}")

    train_times = arrays["train"][2]
    valid_times = arrays["validation"][2]
    test_times = arrays["test"][2]
    assert train_times.max() < valid_times.min() < test_times.min(), "Target timestamp split leakage"
    train_end = int(len(featured) * TRAIN_FRACTION)
    assert pd.Timestamp(metadata["splits"]["validation"]["target_start"]) == featured["timestamp"].iloc[train_end]
    assert featured["timestamp"].diff().dropna().eq(pd.Timedelta(hours=1)).all(), "Hourly grid is not continuous"
    print("  Checks passed: rank, dimensions, finite arrays, chronological targets, train-only scaler fit, and hourly alignment.")


LAS_VEGAS
Hourly rows: 70,128; features: 33
  train      X=(48753, 168, 33), y=(48753,), targets=2016-01-15 00:00:00 to 2021-08-07 08:00:00
  validation X=(10519, 168, 33), y=(10519,), targets=2021-08-07 09:00:00 to 2022-10-19 15:00:00
  test       X=(10520, 168, 33), y=(10520,), targets=2022-10-19 16:00:00 to 2023-12-31 23:00:00
  Checks passed: rank, dimensions, finite arrays, chronological targets, train-only scaler fit, and hourly alignment.

PHOENIX
Hourly rows: 70,128; features: 33
  train      X=(48753, 168, 33), y=(48753,), targets=2016-01-15 00:00:00 to 2021-08-07 08:00:00
  validation X=(10519, 168, 33), y=(10519,), targets=2021-08-07 09:00:00 to 2022-10-19 15:00:00
  test       X=(10520, 168, 33), y=(10520,), targets=2022-10-19 16:00:00 to 2023-12-31 23:00:00
  Checks passed: rank, dimensions, finite arrays, chronological targets, train-only scaler fit, and hourly alignment.


## Output files

Each city gets a folder under `D:\Downloads\BTP\model_training\lstm_training_3\data\tcn_prepared`:

- `X_train.npy`, `X_validation.npy`, `X_test.npy` — float32 arrays shaped `(samples, 168, features)`.
- `y_train.npy`, `y_validation.npy`, `y_test.npy` — scaled one-hour-ahead load targets.
- `feature_scaler.joblib`, `target_scaler.joblib` — fit on training rows only; use the target scaler to invert predictions.
- `*_target_timestamps.csv` — target timestamp for every sequence.
- `featured_hourly_data.csv` — unscaled engineered hourly table for auditing.
- `metadata.json` — features, split boundaries, shapes/sample counts, and source files.

A TCN can directly consume `X_train.npy` as `(batch, time_steps, channels)`. Keep the feature order in `metadata.json` unchanged when fitting and predicting.